### Combined Pipeline

Imports

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_percentage_error

df = pd.read_csv("../Processed_data/Unified_dataset/unified_dataset.csv")
df["task_type"] = df["process"].str.split(":").str[-1].str.upper()
print("Loaded:", len(df), "tasks")

Loaded: 22672 tasks


Hardware-Aware Models

In [3]:
base_features = ["rchar", "cpus", "cores", "ram", "cpu_benchmark", "io_read", "io_write"]
hu = df[df["source_dataset"] == "augur_hu"].copy()
hu_encoded = pd.get_dummies(hu, columns=["task_type"], prefix="task")
task_cols = [c for c in hu_encoded.columns if c.startswith("task_")]
feature_cols = base_features + task_cols

def evaluate_target(data, target):
    d = data.dropna(subset=[target] + base_features)
    d = d[d[target] > 0]
    X, y = d[feature_cols], d[target]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    m = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)
    print(f"  {target:12s} R2={r2_score(yte, pred):.3f}  MAPE={mean_absolute_percentage_error(yte, pred)*100:.1f}%")
    return m

print("Hardware-aware models:")
models = {t: evaluate_target(hu_encoded, t) for t in ["runtime_s", "peak_mem", "energy_j"]}

Hardware-aware models:
  runtime_s    R2=0.989  MAPE=14.5%
  peak_mem     R2=0.984  MAPE=15.5%
  energy_j     R2=0.957  MAPE=17.4%


Task-Only Energy Model (cross-vendor, pooled servers)

In [4]:
_train = df[df["source_dataset"].isin(["augur_gu", "augur_hu"])].copy()
_train_enc = pd.get_dummies(_train, columns=["task_type"], prefix="task")
_task_features = ["rchar", "cpus", "concurrent_task_count"]
_tcols = [c for c in _train_enc.columns if c.startswith("task_")]
_feat = _task_features + _tcols
_valid = _train_enc.dropna(subset=["energy_j"] + _task_features)
_valid = _valid[_valid["energy_j"] > 0]

energy_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
energy_model.fit(_valid[_feat], _valid["energy_j"])
print("Task-only energy_model ready:", energy_model.feature_names_in_.tolist()[:4], "...")

Task-only energy_model ready: ['rchar', 'cpus', 'concurrent_task_count', 'task_ANNOTATE_BOOLEAN_PEAKS'] ...


process_uprof_run

In [5]:
def process_uprof_run(timechart_path, trace_path, output_path,
                      source_dataset, source_workflow, node,
                      tz_offset_hours=1, trace_sep=","):
    with open(timechart_path) as f:
        lines = f.readlines()
    hdr = next(i for i, l in enumerate(lines) if l.startswith("RecordId"))
    up = pd.read_csv(timechart_path, skiprows=hdr, encoding="latin-1")
    up.columns = [c.strip() for c in up.columns]
    up = up.dropna(subset=["Timestamp"])

    def to_sec(t):
        h, m, s, ms = map(int, str(t).split(":"))
        return h*3600 + m*60 + s + ms/1000
    up["t_sec"] = up["Timestamp"].apply(to_sec)
    up["pkg_w"] = pd.to_numeric(up["socket0-package-power"], errors="coerce")

    tr = pd.read_csv(trace_path, sep=trace_sep, engine="python", quoting=3)
    tr.columns = [c.strip().strip('"') for c in tr.columns]
    tr = tr[tr["status"] == "COMPLETED"].copy()

    for c in ["start", "complete"]:
        dt = pd.to_datetime(pd.to_numeric(tr[c], errors="coerce"), unit="ms") + pd.Timedelta(hours=tz_offset_hours)
        tr[c + "_sec"] = dt.dt.hour*3600 + dt.dt.minute*60 + dt.dt.second + dt.dt.microsecond/1e6

    tr["task_type"] = tr["process"].str.split(":").str[-1].str.upper()

    covers = (up["t_sec"].min() <= tr["start_sec"].min()) and (up["t_sec"].max() >= tr["complete_sec"].max())
    print(f"[{source_workflow}@{node}] uProf covers workflow: {covers}")

    def energy(row):
        w = up[(up["t_sec"] >= row["start_sec"]) & (up["t_sec"] <= row["complete_sec"])]
        return w["pkg_w"].sum() if len(w) else np.nan
    tr["energy_j"] = tr.apply(energy, axis=1)

    if pd.api.types.is_numeric_dtype(tr.get("realtime")):
        tr["runtime_s"] = pd.to_numeric(tr["realtime"], errors="coerce")
    else:
        dt = pd.to_datetime(tr["complete"]) - pd.to_datetime(tr["start"])
        tr["runtime_s"] = dt.dt.total_seconds() * 1000

    def parse_mem(s):
        if pd.isna(s): return np.nan
        s = str(s).strip(); parts = s.split()
        if len(parts) < 2:
            return pd.to_numeric(s, errors="coerce")
        num = float(parts[0]); unit = parts[1].upper()
        return num * {"B":1,"KB":1024,"MB":1048576,"GB":1073741824}.get(unit, np.nan)
    tr["peak_mem"] = tr["peak_rss"].apply(parse_mem)

    tr["source_dataset"] = source_dataset
    tr["source_workflow"] = source_workflow
    tr["node"] = node
    tr = tr[tr["energy_j"] > 0].copy()
    print(f"  Tasks with energy: {len(tr)}")
    tr.to_csv(output_path, index=False)
    return tr

In [6]:
def process_rapl_run(rapl_path, trace_path, output_path, node, workflow,
                     RAPL_MAX_UJ=262143328850):
    rapl = pd.read_csv(rapl_path, header=None, names=["ts_ms","energy_uj"])
    rapl = rapl.sort_values("ts_ms").reset_index(drop=True)
    rapl["diff_uj"] = rapl["energy_uj"].diff()
    rapl.loc[rapl["diff_uj"] < 0, "diff_uj"] += RAPL_MAX_UJ
    rapl["energy_j"] = rapl["diff_uj"] / 1e6
    rapl = rapl.dropna(subset=["energy_j"])
    rapl = rapl[rapl["energy_j"] >= 0]

    tr = pd.read_csv(trace_path)
    tr = tr[tr["status"] == "COMPLETED"].copy()
    tr["start"] = pd.to_numeric(tr["start"], errors="coerce")
    tr["complete"] = pd.to_numeric(tr["complete"], errors="coerce")

    def energy(s, c):
        m = (rapl["ts_ms"] >= s) & (rapl["ts_ms"] <= c)
        return rapl.loc[m, "energy_j"].sum()
    tr["energy_j"] = tr.apply(lambda r: energy(r["start"], r["complete"]), axis=1)
    tr["runtime_s"] = pd.to_numeric(tr["realtime"], errors="coerce")   # ms
    tr["peak_mem"] = pd.to_numeric(tr["peak_rss"], errors="coerce")    # bytes
    tr["task_type"] = tr["process"].str.split(":").str[-1].str.upper()
    tr["source_dataset"] = node
    tr["source_workflow"] = workflow
    tr["node"] = node
    tr = tr[tr["energy_j"] > 0].copy()
    print(f"[{workflow}@{node}] tasks with energy: {len(tr)}")
    tr.to_csv(output_path, index=False)
    return tr


predict_and_save

In [7]:
def predict_and_save(measured_df, output_path, label):
    d = measured_df.copy()
    enc = pd.get_dummies(d, columns=["task_type"], prefix="task")
    X = enc.reindex(columns=energy_model.feature_names_in_, fill_value=0)
    d["predicted_energy_j"] = energy_model.predict(X)
    X = enc.reindex(columns=models["runtime_s"].feature_names_in_, fill_value=0)
    d["predicted_runtime_s"] = models["runtime_s"].predict(X)
    X = enc.reindex(columns=models["peak_mem"].feature_names_in_, fill_value=0)
    d["predicted_peak_mem"] = models["peak_mem"].predict(X)

    def report(meas, pred, name):
        v = d[(d[meas] > 0) & (d[pred] > 0)].dropna(subset=[meas, pred])
        if len(v) < 3:
            print(f"  {name}: too few ({len(v)})"); return
        rho = spearmanr(v[meas], v[pred]).correlation
        ratio = (v[pred] / v[meas]).median()
        print(f"  {name:14s} (n={len(v)}): Spearman {rho:.3f} | ratio {ratio:.2f}x")

    print(f"\n=== {label} ===")
    report("energy_j",  "predicted_energy_j",  "ENERGY (J)")
    report("runtime_s", "predicted_runtime_s", "RUNTIME (ms)")
    report("peak_mem",  "predicted_peak_mem",  "MEMORY (bytes)")
    d.to_csv(output_path, index=False)
    return d

train_and_test (Augmentation Experiment)

In [8]:
def train_and_test(train_df, test_df, target, label):
    tr = train_df.dropna(subset=[target, "rchar"]).copy()
    tr = tr[tr[target] > 0].reset_index(drop=True)
    te = test_df.dropna(subset=[target, "rchar"]).copy()
    te = te[te[target] > 0].reset_index(drop=True)
    n_tr = len(tr)
    both = pd.concat([tr, te], ignore_index=True)
    enc = pd.get_dummies(both, columns=["task_type"], prefix="task")
    feat = ["rchar", "cpus", "concurrent_task_count"] + [c for c in enc.columns if c.startswith("task_")]
    Xtr, ytr = enc.iloc[:n_tr][feat], tr[target].values
    Xte, yte = enc.iloc[n_tr:][feat], te[target].values
    m = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)
    ratio = np.median(pred / yte)
    rho = spearmanr(yte, pred).correlation
    print(f"  {label:22s}: train n={n_tr}, test n={len(yte)} | ratio {ratio:.2f}x | Spearman {rho:.3f}")
    return ratio, rho

In [ ]:
combined = pd.read_csv("../Processed_data/Augmented_combined_dataset/new_combined_dataset.csv")
servers = combined[combined["source_dataset"].str.startswith("augur")]
amd = combined[combined["node"] == "amd_ryzen_7"]
intel = combined[combined["node"] == "intel_i5"]
r5 = combined[combined["node"] == "amd_ryzen_5"]
for target in ["energy_j", "runtime_s", "peak_mem"]:
    print(f"{target} - held-out Ryzen 5")
    train_and_test(servers, r5, target, "servers only")
    train_and_test(pd.concat([servers, amd, intel]), r5, target, "servers+AMD+Intel")

In [ ]:
for target in ["energy_j", "runtime_s", "peak_mem"]:
    print(f"{target} - held-out Ryzen 7")
    train_and_test(servers, amd, target, "servers only")
    train_and_test(pd.concat([servers, r5, intel]), amd, target, "servers+AMD+Intel")

In [ ]:
for target in ["energy_j", "runtime_s", "peak_mem"]:
    print(f"{target} - held-out intel i5")
    train_and_test(servers, intel, target, "servers only")
    train_and_test(pd.concat([servers, r5, amd]), intel, target, "servers+AMD+Intel")